# P08_RAG_Evaluation — RAG: поиск и ответы / Retrieval and answers

**Источник / Source:** [SQuAD 2.0](https://rajpurkar.github.io/SQuAD-explorer/), исходный `dev-v2.0.json`, CC BY-SA 4.0. Поднабор сохраняет тексты абзацев и разметку без изменений; алгоритм выбора и SHA-256 исходника находятся в JSON. / Paragraphs and annotations are unchanged; the JSON records selection rules and the original SHA-256.

**Наблюдение / Observation:** вопрос человека к абзацу Wikipedia. Признак — текст вопроса; доступная база — тексты абзацев. Цель поиска — ID исходного абзаца; цель ответа — один из размеченных текстов. / A human question about a Wikipedia paragraph. Input: question text and a fixed paragraph collection. Retrieval target: source paragraph ID. Answer target: an annotated text span.

**Разбиение / Split:** первые две статьи — Validation (12 вопросов), следующие две — Test (12). Все 48 документов доступны заранее: проверяется новый вопрос к известной базе, а не перенос на новую базу. Не добавляйте вопросы/эталоны в индекс. Соседние вопросы коррелированы; малая выборка не является полным benchmark. Пересечение с обучением FLAN-T5 неизвестно. / First two articles: validation (12 questions); next two: test (12). All 48 documents are available in advance. This tests new questions over a known collection, not transfer to a new collection. Do not index questions or reference answers. Questions are correlated; this small subset is not a full benchmark. FLAN-T5 training overlap is unknown.

**CPU Colab:** API-ключ не нужен. Первый запуск скачивает модель; при запросе Colab перезапустите среду после установки. Все ответы модели — на английском, поскольку исходные вопросы английские. Задания и пояснения полностью RU/EN. / No API key. The first run downloads the model. Restart after installation if Colab requests it. Model answers are English because source questions are English. Instructions and explanations are bilingual.


In [ ]:
import sys, subprocess # Найти текущий интерпретатор / Locate the current interpreter.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3', 'sentencepiece==0.2.2', 'scikit-learn', 'pandas']) # Установить зависимости CPU / Install CPU dependencies.


In [ ]:
STUDENT_NAME = '' # Введите ФИО / Enter your full name.
STUDENT_GROUP = '' # Введите группу / Enter your group.


In [ ]:
import json, urllib.request, hashlib, re, string # Загрузка, проверка источника и нормализация / Loading, source verification and normalization.
from collections import Counter # Пересечение токенов с учётом повторов / Token overlap with multiplicity.
import numpy as np, pandas as pd, torch # Массивы, отчёты и CPU-модель / Arrays, reports and CPU model.
from sklearn.feature_extraction.text import TfidfVectorizer # Лексическое представление документов / Lexical document representation.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Токенизатор и генератор / Tokenizer and generator.
DATA_URL = 'https://raw.githubusercontent.com/alexander-toschev/ml-cs-intro/main/data/rag/practice-squad2.json' # Публичный реальный поднабор SQuAD / Public real SQuAD subset.
raw = urllib.request.urlopen(DATA_URL).read() # Загрузить фиксированный файл / Download the fixed file.
assert hashlib.sha256(raw).hexdigest() == '0fa43c712765a54c3fcf0fdef10124a5ad854afda1342bd2535b882f2aa1e14b' # Проверить неизменность поднабора / Verify subset integrity.
data = json.loads(raw) # Прочитать документы и разметку / Read documents and annotations.
docs = data['documents'] # Использовать только тексты абзацев в индексе / Index paragraph texts only.
by_id = {d['id']: d for d in docs} # Быстрый доступ к источнику по ID / Access sources by ID.
val = [q for q in data['questions'] if q['split'] == 'val'] # Вопросы для выбора настроек / Questions for selecting settings.
test_rows = [q for q in data['questions'] if q['split'] == 'test'] # Отложенные вопросы, пока не оценивать / Held-out questions, not yet evaluated.
assert not ({q['article'] for q in val} & {q['article'] for q in test_rows}) # Проверить разделение по статьям / Verify article-disjoint questions.
vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm='l2') # Фиксированный unigram/bigram baseline / Fixed unigram/bigram baseline.
D = vectorizer.fit_transform([d['text'] for d in docs]) # IDF по заранее известной базе, без вопросов / Fit IDF on the known collection, without questions.
doc_ids = [d['id'] for d in docs] # Сохранить соответствие строк и документов / Preserve matrix row-to-document mapping.
val_scores = (vectorizer.transform([q['question'] for q in val]) @ D.T).toarray() # Косинусные score validation / Validation cosine scores.
print(len(docs), len(val), len(test_rows), data['source']) # Показать объём и происхождение / Show sizes and provenance.


## Блок 1 / Block 1: Ранжирование / Ranking

TF-IDF сравнивает совпадающие слова и биграммы; после L2-нормировки скалярное произведение равно cosine similarity. / TF-IDF compares word and bigram overlap; after L2 normalization the dot product is cosine similarity.

### Пример / Worked example

In [ ]:
best = int(np.argmax(val_scores[0])) # Найти один максимальный score без правила равенств / Find one maximum without a tie rule.
print(val[0]['question'], docs[best]['title'], float(val_scores[0, best])) # Показать вопрос и найденную статью / Show the question and retrieved article.


### Задание 1 / Exercise 1

Реализуйте `rank_documents(scores, ids, k)`: scores — конечный 1D массив; ids — уникальные строки той же длины; k — целое ≥0. Верните список ID по убыванию score, при равенстве — ID по возрастанию. Не изменяйте входы. k=0 даёт [], k больше корпуса — все документы. / Implement `rank_documents(scores, ids, k)`: finite 1D scores, equally sized unique string IDs, integer k≥0. Return IDs by descending score, breaking ties by ascending ID. Do not mutate inputs. k=0 returns []; k above collection size returns all documents.

In [ ]:
def rank_documents(scores, ids, k): # Ранжировать документы с явным разрешением равенств / Rank documents with explicit tie breaking.
    raise NotImplementedError('TODO: implement / реализуйте') # Напишите собственное решение / Write your own implementation.


### Автоматическая проверка / Executable check

In [ ]:
rankings = [rank_documents(s, doc_ids, 3) for s in val_scores] # Получить top-3 для реальных вопросов / Retrieve top three for real questions.
expected = [doc_ids[i] for i in np.lexsort((np.array(doc_ids), -val_scores[0]))[:3]] # Независимый порядок через NumPy / Independent NumPy ordering.
assert rankings[0] == expected and len(rankings) == len(val) # Проверить порядок и число запросов / Check ordering and query count.
assert rank_documents(val_scores[0], doc_ids, 0) == [] # Нулевой бюджет возвращает пустой список / A zero budget returns an empty list.
assert len(rank_documents(val_scores[0], doc_ids, 999)) == len(docs) # Не выдумывать отсутствующие документы / Do not invent extra documents.
zeros = np.zeros_like(val_scores[0]) # Смоделировать отсутствие лексических совпадений для тех же документов / Model no lexical overlap for the same documents.
assert rank_documents(zeros, doc_ids, 3) == sorted(doc_ids)[:3] # Проверить детерминированное равенство / Check deterministic ties.
print('Exercise 1: OK', rankings[0]) # Подтвердить проверку / Confirm the check.


## Блок 2 / Block 2: Метрики поиска / Retrieval metrics

Hit@k проверяет попадание исходного абзаца. MRR@k усредняет обратный ранг с нулём при отсутствии. Другие релевантные абзацы могут быть не размечены. / Hit@k checks source-paragraph inclusion. MRR@k averages reciprocal rank, using zero for missing references. Other relevant paragraphs may be unannotated.

### Пример / Worked example

In [ ]:
print(val[0]['gold_id'] in rankings[0]) # Проверить одно реальное попадание / Check one real retrieval hit.
print(list(enumerate(rankings[0], start=1))) # Показать ранги с единицы / Show one-based ranks.


### Задание 2 / Exercise 2

Реализуйте `retrieval_report(rankings, gold_ids, k)`. Входы непустые и одинаковой длины, ID внутри выдачи уникальны. Ограничьте каждый список первыми k. Верните dict с float `hit`, float `mrr`, int `n`. Ранг начинается с 1; если эталон отсутствует, вклад в обе метрики 0. При k=0 обе метрики 0. / Implement `retrieval_report(rankings, gold_ids, k)`. Inputs are nonempty and equally sized; each ranking has unique IDs. Evaluate the first k entries. Return float `hit`, float `mrr`, and int `n`. Ranks start at 1; missing references contribute zero to both metrics. k=0 gives zero metrics.

In [ ]:
def retrieval_report(rankings, gold_ids, k): # Оценить найденные абзацы на заданной глубине / Evaluate paragraph retrieval at a fixed depth.
    raise NotImplementedError('TODO: implement / реализуйте') # Напишите собственное решение / Write your own implementation.


### Автоматическая проверка / Executable check

In [ ]:
gold_ids = [q['gold_id'] for q in val] # Эталонные абзацы validation / Validation reference paragraphs.
for k in [1, 3]: # Проверить обе заранее заданные глубины / Check both predefined depths.
    report = retrieval_report(rankings, gold_ids, k) # Вызвать решение студента / Call the student implementation.
    hits = [g in r[:k] for r, g in zip(rankings, gold_ids)] # Проверить попадания непосредственно / Directly check hits.
    rr = [sum((doc == g) / (i + 1) for i, doc in enumerate(r[:k])) for r, g in zip(rankings, gold_ids)] # Независимо суммировать обратные ранги / Independently sum reciprocal ranks.
    assert report['n'] == len(val) and np.isclose(report['hit'], np.mean(hits)) # Сравнить объём и Hit / Compare sample size and Hit.
    assert np.isclose(report['mrr'], np.mean(rr)) # Сравнить MRR / Compare MRR.
    print(k, report) # Показать реальные результаты / Show actual results.
assert retrieval_report(rankings, gold_ids, 0)['hit'] == 0 # Пустая выдача не содержит эталон / An empty ranking cannot contain the reference.


## Блок 3 / Block 3: Бюджет контекста / Context budget

Считаем токены полного запроса, а не длину строки. Длинный абзац пропускаем; следующий всё ещё можно добавить. / Count tokens of the entire prompt, not characters. Skip a long paragraph but still consider later passages.

In [ ]:
MODEL_ID = 'google/flan-t5-small' # Компактная instruction-tuned модель / Compact instruction-tuned model.
MODEL_REV = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab' # Фиксированная версия весов / Fixed weights revision.
torch.set_num_threads(2) # Ограничить нагрузку CPU / Limit CPU load.
torch.manual_seed(2026) # Зафиксировать генератор / Seed the generator.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV) # Использовать соответствующий токенизатор / Use the matching tokenizer.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, revision=MODEL_REV, use_safetensors=True).to('cpu').eval() # Загрузить веса и отключить dropout / Load weights and disable dropout.
def make_prompt(question, selected): # Единый формат для проверки бюджета / Shared format for budget checking.
    context = '\n'.join('[' + d['id'] + '] ' + d['text'] for d in selected) # Сохранить ID возле каждого источника / Keep each source ID beside its text.
    return 'Answer the question using only the context. Give a short answer. If the context does not answer it, say UNANSWERABLE.\nContext:\n' + context + '\nQuestion: ' + question + '\nAnswer:' # Задать формат и правило отказа / Specify output and abstention rules.


### Пример / Worked example

In [ ]:
example_prompt = make_prompt(val[0]['question'], [by_id[rankings[0][0]]]) # Собрать пример с одним источником / Build a one-source example.
print(len(tokenizer.encode(example_prompt)), example_prompt[:220]) # Показать длину и начало запроса / Show token count and prompt prefix.


### Задание 3 / Exercise 3

Реализуйте `pack_context(question, ranked_ids, budget=480)`. Используйте заданный `make_prompt`. Если инструкция и вопрос без документов превышают budget, вызовите ValueError. Иначе проходите ID по порядку: добавляйте целый абзац, только если полный новый запрос помещается. Не помещается — пропустите и продолжайте. Верните `(prompt, included_ids)`. Не обрезайте вопрос или абзац. / Implement `pack_context(question, ranked_ids, budget=480)` using the provided `make_prompt`. Raise ValueError if instruction and question alone exceed the budget. Otherwise consider IDs in order and include each whole paragraph only if the full new prompt fits. Skip non-fitting passages and continue. Return `(prompt, included_ids)`. Never truncate the question or a paragraph.

In [ ]:
def pack_context(question, ranked_ids, budget=480): # Упаковать целые абзацы в токенный бюджет / Pack whole passages within a token budget.
    raise NotImplementedError('TODO: implement / реализуйте') # Напишите собственное решение / Write your own implementation.


### Автоматическая проверка / Executable check

In [ ]:
q = val[0]['question'] # Реальный validation-вопрос / A real validation question.
for budget in [160, 480]: # Проверить ограниченный и обычный бюджет / Check constrained and normal budgets.
    prompt, included = pack_context(q, rankings[0], budget) # Собрать запрос решением студента / Assemble the student prompt.
    assert len(tokenizer.encode(prompt)) <= budget and q in prompt # Не превышать бюджет и не терять вопрос / Respect the budget and retain the question.
    assert included == [d for d in rankings[0] if d in included] # Сохранить порядок без новых ID / Preserve ordering without inventing IDs.
    assert all(by_id[d]['text'] in prompt for d in included) # Не обрезать включённые абзацы / Do not truncate included paragraphs.
    manual = [] # Независимо восстановить жадный выбор / Independently reconstruct greedy selection.
    for did in rankings[0]: # Перебрать реальную выдачу / Iterate over actual retrieval results.
        if len(tokenizer.encode(make_prompt(q, [by_id[x] for x in manual + [did]]))) <= budget: # Проверить целый кандидат / Check the complete candidate prompt.
            manual.append(did) # Принять помещающийся ID / Accept the fitting ID.
    assert included == manual and prompt == make_prompt(q, [by_id[x] for x in manual]) # Проверить точный контракт / Check the exact contract.
try: # Проверить невозможный бюджет / Check an impossible budget.
    pack_context(q, rankings[0], 1) # Вопрос не может поместиться / The question cannot fit.
    raise AssertionError('Expected ValueError') # Требовать явную ошибку / Require an explicit error.
except ValueError: # Принять предусмотренный отказ / Accept the specified failure.
    pass # Проверка успешно завершена / The check succeeded.
print('Exercise 3: OK') # Подтвердить упаковку / Confirm packing checks.


## Блок 4 / Block 4: Оценка ответов / Answer evaluation

EM сравнивает нормализованные строки; token F1 учитывает повторения. Максимум берём по допустимым эталонам. Метрики не доказывают grounding. / EM compares normalized strings; token F1 retains multiplicity. Take the best acceptable reference. These metrics do not establish grounding.

In [ ]:
def normalize_answer(text): # Нормализация английского SQuAD / English SQuAD normalization.
    lowered = text.lower() # Игнорировать регистр / Ignore letter case.
    no_punctuation = ''.join(c for c in lowered if c not in string.punctuation) # Удалить ASCII-пунктуацию / Remove ASCII punctuation.
    no_articles = re.sub(r'\b(a|an|the)\b', ' ', no_punctuation) # Удалить английские артикли / Remove English articles.
    return ' '.join(no_articles.split()) # Свести пробелы к одному / Collapse whitespace.
def generate_answer(prompt): # Получить детерминированный короткий ответ / Generate a deterministic short answer.
    encoded = tokenizer(prompt, return_tensors='pt', truncation=False) # Не допускать скрытого усечения / Prevent silent truncation.
    assert encoded['input_ids'].shape[1] <= 480 # Проверить длину перед моделью / Check length before generation.
    with torch.inference_mode(): # Не вычислять градиенты / Disable gradient tracking.
        output = model.generate(**encoded, max_new_tokens=32, do_sample=False) # Greedy decoding с фиксированным пределом / Greedy decoding with a fixed output limit.
    return tokenizer.decode(output[0], skip_special_tokens=True).strip() # Сохранить исходный текст ответа / Preserve the raw answer text.


### Пример / Worked example

In [ ]:
reference = val[0]['answers'][0] # Взять настоящий эталон SQuAD / Take a real SQuAD reference.
print(reference, normalize_answer(reference), Counter(normalize_answer(reference).split())) # Показать нормализацию и кратность / Show normalization and multiplicity.


### Задание 4 / Exercise 4

Реализуйте `score_answer(prediction, references)` без sklearn-метрик. Точный маркер UNANSWERABLE (без учёта регистра и крайних пробелов) означает пустую строку. Пустой список references заменяется одним пустым эталоном. Примените `normalize_answer`, затем EM и F1=2×число общих токенов/(число токенов предсказания+эталона). Считайте кратность! Если оба списка пусты, EM=F1=1; если пуст только один — 0. Верните dict `em`, `f1`, максимум отдельно по эталонам. После реализации запустите validation-эксперимент и проверки. / Implement `score_answer(prediction, references)` without sklearn metrics. The exact UNANSWERABLE marker, ignoring case and surrounding spaces, maps to empty text. Replace an empty reference list by one empty reference. Apply `normalize_answer`, then EM and F1=2×shared-token count/(prediction-token count+reference-token count), retaining multiplicity. Both empty lists score 1; only one empty list scores 0. Return `em` and `f1`, maximizing each over references. Then run the validation experiment and checks.

In [ ]:
def score_answer(prediction, references): # Оценить один ответ по всем допустимым эталонам / Score one answer against all accepted references.
    raise NotImplementedError('TODO: implement / реализуйте') # Напишите собственное решение / Write your own implementation.


**До следующей ячейки запишите прогноз / Before the next cell, record a prediction:** поможет ли k=3 относительно k=1 и почему? Меняем только k; при равном validation token F1 выбираем k=1. Test пока не используем. / Will k=3 improve over k=1, and why? Change only k; equal validation token F1 retains k=1. Do not use test yet.

Прогноз / Prediction: …

In [ ]:
def evaluate(rows, k): # Оценить фиксированный конвейер на заданных вопросах / Evaluate a fixed pipeline on given questions.
    scores = (vectorizer.transform([q['question'] for q in rows]) @ D.T).toarray() # Посчитать лексические близости / Compute lexical similarities.
    logs = [] # Сохранить подробный журнал / Collect detailed logs.
    for q, s in zip(rows, scores): # Обработать каждый реальный вопрос / Process each real question.
        ranked = rank_documents(s, doc_ids, k) # Найти top-k / Retrieve top-k.
        prompt, included = pack_context(q['question'], ranked) # Собрать ограниченный контекст / Assemble bounded context.
        prediction = generate_answer(prompt) # Получить ответ без эталонов в запросе / Generate without references in the prompt.
        metrics = score_answer(prediction, q['answers']) # Оценить ответ после генерации / Score after generation.
        logs.append(dict(id=q['id'], question=q['question'], gold_id=q['gold_id'], ranked=ranked, included=included, prediction=prediction, answers=q['answers'], tokens=len(tokenizer.encode(prompt)), **metrics)) # Сохранить проверяемые свидетельства / Store auditable evidence.
    return logs # Вернуть записи, а не только среднее / Return records rather than only averages.
val_runs = {k: evaluate(val, k) for k in [1, 3]} # Менять только число извлекаемых фрагментов / Change only the number of retrieved passages.
summary = {k: float(np.mean([r['f1'] for r in rows])) for k, rows in val_runs.items()} # Критерий выбора: средний token F1 / Selection metric: mean token F1.
selected_k = max([1, 3], key=lambda k: (summary[k], -k)) # При точном равенстве сохранить k=1 / Retain k=1 on an exact tie.
print('Validation F1:', summary, 'Selected k:', selected_k) # Зафиксировать решение до Test / Record the choice before test.
display(pd.DataFrame(val_runs[selected_k])[['id', 'prediction', 'answers', 'em', 'f1']]) # Показать реальные ответы для анализа / Show actual answers for analysis.


### Автоматическая проверка / Executable check

In [ ]:
for row in val_runs[selected_k]: # Проверить реальные validation-предсказания / Check actual validation predictions.
    result = score_answer(row['prediction'], row['answers']) # Пересчитать оценку / Recompute the score.
    assert 0 <= result['em'] <= 1 and 0 <= result['f1'] <= 1 # Проверить диапазон / Check score bounds.
    p = normalize_answer('' if row['prediction'].strip().upper() == 'UNANSWERABLE' else row['prediction']).split() # Независимо подготовить токены / Independently prepare tokens.
    reference_scores = [] # Собрать контрольные F1 / Collect reference F1 scores.
    for answer in row['answers']: # Проверить каждый допустимый вариант / Check each accepted reference.
        remaining = normalize_answer(answer).split() # Копия эталонных токенов / Copy reference tokens.
        denominator, overlap = len(p) + len(remaining), 0 # Размер мультимножеств и пересечение / Multiset sizes and overlap.
        for token in p: # Сопоставлять каждое вхождение отдельно / Match each occurrence separately.
            if token in remaining: # Проверить ещё не использованное совпадение / Check for an unused match.
                remaining.remove(token) # Не использовать эталонный токен дважды / Do not reuse a reference token.
                overlap += 1 # Засчитать совпадение / Count the match.
        reference_scores.append(2 * overlap / denominator if denominator else 1.0) # Независимая реализация F1 / Independent F1 implementation.
    assert np.isclose(result['f1'], max(reference_scores)) # Проверить фактические оценки / Check actual scores.
    assert result['em'] == max(float(' '.join(p) == normalize_answer(a)) for a in row['answers']) # Независимо проверить EM / Independently check EM.
assert score_answer('UNANSWERABLE', []) == {'em': 1.0, 'f1': 1.0} # Оба пустых ответа совпадают / Two empty answers match.
assert score_answer('', val[0]['answers']) == {'em': 0.0, 'f1': 0.0} # Отказ на ответимом вопросе ошибочен / Abstaining on an answerable question is wrong.
print('Exercise 4: OK') # Подтвердить проверку / Confirm the check.


## Финальный эксперимент / Final experiment

Теперь выбор зафиксирован. Выполните Test один раз; не меняйте настройки по его результату. / Selection is now frozen. Run test once; do not tune using its result.

In [ ]:
test_log = evaluate(test_rows, selected_k) # Единственная финальная оценка выбранного варианта / One final evaluation of the selected configuration.
result = {'selected_k': selected_k, 'validation_f1': summary, 'test_f1': float(np.mean([r['f1'] for r in test_log])), 'test_em': float(np.mean([r['em'] for r in test_log])), 'model_revision': MODEL_REV, 'data_sha256': hashlib.sha256(raw).hexdigest(), 'validation': val_runs, 'test': test_log} # Сохранить параметры и свидетельства / Preserve settings and evidence.
with open('RAG_practice_experiment.json', 'w') as f: # Создать воспроизводимый журнал / Create a reproducible log.
    json.dump(result, f, ensure_ascii=False, indent=2) # Записать без потери исходных ответов / Save without losing raw answers.
print('Test:', result['test_em'], result['test_f1'], 'n=', len(test_log)) # Показать качество и объём / Report quality and sample size.


## Анализ и отчёт / Analysis and report

1. **Поиск (5 баллов в ДЗ):** запишите Hit@1, Hit@3, MRR@3 и число вопросов. Объясните разницу между Hit и MRR. / **Retrieval (5 homework points):** report Hit@1, Hit@3, MRR@3 and n; explain Hit versus MRR.
2. **Ошибка (5):** найдите один validation-вопрос с F1<1 (если нет, явно сообщите). Укажите ID, вопрос, ranked/included ID, ответ и эталоны. Разделите ошибку поиска, упаковки и генерации; проверьте текст источника. / **Error (5):** inspect one validation question with F1<1, or explicitly report none. Include its ID, question, ranked/included IDs, prediction and references. Distinguish retrieval, packing and generation failures; inspect the source.
3. **Выбор (5):** приведите заранее записанную гипотезу, оба validation F1, selected k, test EM/F1 и n. Объясните разрыв без подбора по Test. / **Selection (5):** report the prior hypothesis, both validation F1 scores, selected k, test EM/F1 and n. Explain the gap without test tuning.
4. **Ограничения (5):** почему найденный ID не доказывает, что ответ опирается на источник? Почему неответимость SQuAD относится к абзацу? Укажите ограничения поднабора, источники/AI-помощь и серверный источник времени сдачи (для ДЗ). / **Limitations (5):** why does a retrieved ID not prove grounding? Why is SQuAD unanswerability paragraph-specific? State subset limitations, sources/AI assistance, and the server submission-time source for homework.

Ответ / Your response: …
